# Imports

In [14]:
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# Load Raw Sentiment File

In [5]:
df_sent = pd.read_excel(r"..\..\Data\Main\Bloomberg_Raw\Sentiment_Raw_Bloomberg_Serial_Dates.xlsx")
print(f"Shape: {df_sent.shape}")
print(f"Columns (first 10): {list(df_sent.columns[:10])}")
print(df_sent.head(3))

Shape: (1564, 1122)
Columns (first 10): ['A_date', 'A_sentiment', 'AAL_date', 'AAL_sentiment', 'AAP_date', 'AAP_sentiment', 'AAPL_date', 'AAPL_sentiment', 'ABBV_date', 'ABBV_sentiment']
    A_date  A_sentiment  AAL_date  AAL_sentiment  AAP_date  AAP_sentiment  \
0  43832.0       0.0000   43832.0         0.0421   43836.0         0.0000   
1  43833.0       0.0000   43833.0        -0.0135   43837.0         0.7563   
2  43837.0       0.3292   43836.0         0.0000   43838.0         0.6321   

   AAPL_date  AAPL_sentiment  ABBV_date  ABBV_sentiment  ...  YUM_date  \
0      43832          0.0091    43832.0          0.1624  ...   43832.0   
1      43833          0.0255    43833.0          0.1275  ...   43833.0   
2      43836          0.0256    43836.0          0.0363  ...   43836.0   

   YUM_sentiment  ZBH_date  ZBH_sentiment  ZBRA_date  ZBRA_sentiment  \
0            0.0   43837.0         0.0000    43837.0          0.0000   
1            0.0   43838.0         0.7931    43838.0         -0.

In [6]:
bf_brk = pd.read_excel(r"..\..\Data\Main\Bloomberg_Raw\BF_B_BRK_B_Sentiment.xlsx")
print(f"Shape: {bf_brk.shape}")
print(f"Columns: {list(bf_brk.columns)}")
print(bf_brk.head())

Shape: (1456, 5)
Columns: ['BF-B_date', 'BF-B_sentiment', 'Unnamed: 2', 'BRK-B_date', 'BRK-B_sentiment']
   BF-B_date  BF-B_sentiment  Unnamed: 2  BRK-B_date  BRK-B_sentiment
0    43832.0          0.0000         NaN       43832           0.0000
1    43839.0          0.0000         NaN       43833           0.1031
2    43845.0          0.7876         NaN       43836           0.0000
3    43854.0          0.0000         NaN       43837           0.8647
4    43859.0          0.0000         NaN       43838           0.0000


# Merge into one Sentiment File

In [7]:
# Drop the empty column
bf_brk = bf_brk.drop(columns=['Unnamed: 2'])

# The main dataframe has 1564 rows, BF-B/BRK-B have 1456
# They have different lengths because different tickers have different numbers of trading days
# We need to pad the shorter columns with NaN to match the main dataframe's row count

rows_needed = len(df_sent)
for col in ['BF-B_date', 'BF-B_sentiment', 'BRK-B_date', 'BRK-B_sentiment']:
    values = bf_brk[col].tolist()
    # Pad with NaN if shorter
    values.extend([np.nan] * (rows_needed - len(values)))
    df_sent[col] = values

print(f"Updated shape: {df_sent.shape}")
print(f"Total tickers: {len([c for c in df_sent.columns if c.endswith('_sentiment')])}")

# Verify BF-B and BRK-B are now in the dataframe
print(f"\nBF-B sentiment values: {df_sent['BF-B_sentiment'].notna().sum()}")
print(f"BRK-B sentiment values: {df_sent['BRK-B_sentiment'].notna().sum()}")

Updated shape: (1564, 1122)
Total tickers: 561

BF-B sentiment values: 1080
BRK-B sentiment values: 1456


# Convert Excel Serial Date Numbers to Proper Dates

In [8]:
def excel_to_date(val):
    try:
        num = float(val)
        if 1 < num < 100000:
            return datetime(1899, 12, 30) + timedelta(days=int(num))
        return None
    except:
        return None

date_cols = [c for c in df_sent.columns if c.endswith("_date")]
print(f"Date columns to convert: {len(date_cols)}")

for col in date_cols:
    df_sent[col] = df_sent[col].apply(excel_to_date)

print("Done. Checking AAPL dates:")
print(df_sent["AAPL_date"].dropna().head())
print()
print(df_sent["AAPL_date"].dropna().tail())

Date columns to convert: 561
Done. Checking AAPL dates:
0   2020-01-02
1   2020-01-03
2   2020-01-06
3   2020-01-07
4   2020-01-08
Name: AAPL_date, dtype: datetime64[ns]

1559   2025-12-25
1560   2025-12-26
1561   2025-12-29
1562   2025-12-30
1563   2025-12-31
Name: AAPL_date, dtype: datetime64[ns]


# Validate Dates and Sentiment Values

In [9]:
# Check date ranges
all_dates = []
for col in date_cols:
    vals = df_sent[col].dropna()
    all_dates.extend(vals.tolist())

all_dates = pd.Series(all_dates)
print(f"=== DATE CHECKS ===")
print(f"Total date columns: {len(date_cols)}")
print(f"Total date values: {len(all_dates)}")
print(f"Earliest date: {all_dates.min()}")
print(f"Latest date: {all_dates.max()}")
print(f"Dates before 2020: {(all_dates < '2020-01-01').sum()}")
print(f"Dates after 2025: {(all_dates > '2025-12-31').sum()}")

# Check sentiment values
sent_cols = [c for c in df_sent.columns if c.endswith("_sentiment")]
all_sent = []
for col in sent_cols:
    vals = pd.to_numeric(df_sent[col], errors='coerce').dropna()
    all_sent.extend(vals.tolist())

all_sent = pd.Series(all_sent)
print(f"\n=== SENTIMENT CHECKS ===")
print(f"Total sentiment columns: {len(sent_cols)}")
print(f"Total sentiment values: {len(all_sent)}")
print(f"Min: {all_sent.min()}")
print(f"Max: {all_sent.max()}")
print(f"Mean: {all_sent.mean():.4f}")
print(f"Values outside -1 to +1 range: {((all_sent < -1) | (all_sent > 1)).sum()}")

# Verify BF-B and BRK-B specifically
print(f"\n=== BF-B / BRK-B CHECK ===")
for t in ['BF-B', 'BRK-B']:
    dates = df_sent[f'{t}_date'].dropna()
    sent = df_sent[f'{t}_sentiment'].dropna()
    non_zero = (sent != 0).sum()
    print(f"{t}: {dates.min().date()} to {dates.max().date()}, {len(dates)} days, {non_zero} non-zero sentiment")

=== DATE CHECKS ===
Total date columns: 561
Total date values: 609750
Earliest date: 2020-01-02 00:00:00
Latest date: 2025-12-31 00:00:00
Dates before 2020: 0
Dates after 2025: 0

=== SENTIMENT CHECKS ===
Total sentiment columns: 561
Total sentiment values: 609750
Min: -1.0
Max: 1.0
Mean: -0.0077
Values outside -1 to +1 range: 0

=== BF-B / BRK-B CHECK ===
BF-B: 2020-01-02 to 2025-12-31, 1080 days, 862 non-zero sentiment
BRK-B: 2020-01-02 to 2025-12-31, 1456 days, 1112 non-zero sentiment


# Identify Zero Sentiment Tickers

In [10]:
sent_cols = [c for c in df_sent.columns if c.endswith('_sentiment')]

zero_tickers = []
for col in sent_cols:
    ticker = col.replace('_sentiment', '')
    if (df_sent[col] == 0).all() or df_sent[col].isna().all():
        zero_tickers.append(ticker)

print(f"Tickers with zero sentiment: {len(zero_tickers)}")
print(zero_tickers)

# Save for reference
zero_df = pd.DataFrame({'ticker': zero_tickers})
zero_df.to_csv(r"..\..\Data\Main\zero_sentiment_tickers.csv", index=False)

Tickers with zero sentiment: 14
['AMTM', 'APP', 'DDOG', 'DTM', 'EMBC', 'EME', 'FTRE', 'HOOD', 'IBKR', 'PSKY', 'SLVM', 'TTD', 'VNT', 'XYZ']


# Load Zero Sentiment Tickers from Previous Step

In [12]:
zero_sent = pd.read_csv(r"..\..\Data\Main\zero_sentiment_tickers.csv")
missing_tickers = zero_sent['ticker'].tolist()
print(f"Zero sentiment tickers to investigate: {len(missing_tickers)}")
print(missing_tickers)

Zero sentiment tickers to investigate: 14
['AMTM', 'APP', 'DDOG', 'DTM', 'EMBC', 'EME', 'FTRE', 'HOOD', 'IBKR', 'PSKY', 'SLVM', 'TTD', 'VNT', 'XYZ']


# Check Each Ticker Against All 24 Quarterly Bloomberg Snapshots

In [15]:
folder = r"..\..\Data\Main\Quarterly"
files = sorted(glob.glob(os.path.join(folder, "SPX as of*.xlsx")))
print(f"Found {len(files)} snapshot files\n")

ticker_found_in = {t: [] for t in missing_tickers}

for f in files:
    snapshot = pd.read_excel(f)
    fname = os.path.basename(f)
    
    for t in missing_tickers:
        match = snapshot["Ticker"].str.startswith(t.replace("-", "/"))
        if match.any():
            ticker_found_in[t].append(fname)

print("=== Results ===\n")
for t in missing_tickers:
    if ticker_found_in[t]:
        print(f"{t}: Found in {len(ticker_found_in[t])} snapshots")
        for f in ticker_found_in[t]:
            print(f"   - {f}")
    else:
        print(f"{t}: NOT in any snapshot")
    print()

Found 24 snapshot files

=== Results ===

AMTM: Found in 1 snapshots
   - SPX as of Oct 01 20241.xlsx

APP: Found in 1 snapshots
   - SPX as of Oct 01 20251.xlsx

DDOG: Found in 1 snapshots
   - SPX as of Oct 01 20251.xlsx

DTM: Found in 1 snapshots
   - SPX as of Jul 01 20211.xlsx

EMBC: Found in 1 snapshots
   - SPX as of Apr 01 20221.xlsx

EME: Found in 1 snapshots
   - SPX as of Oct 01 20251.xlsx

FTRE: Found in 1 snapshots
   - SPX as of Jul 03 20231.xlsx

HOOD: Found in 1 snapshots
   - SPX as of Oct 01 20251.xlsx

IBKR: Found in 1 snapshots
   - SPX as of Oct 01 20251.xlsx

PSKY: Found in 1 snapshots
   - SPX as of Oct 01 20251.xlsx

SLVM: Found in 1 snapshots
   - SPX as of Oct 01 20211.xlsx

TTD: Found in 1 snapshots
   - SPX as of Oct 01 20251.xlsx

VNT: Found in 1 snapshots
   - SPX as of Jan 04 20211.xlsx

XYZ: Found in 1 snapshots
   - SPX as of Oct 01 20251.xlsx



## Findings

All 14 zero-sentiment tickers appear in only 1 quarterly snapshot each. These are companies that briefly entered the S&P 500 index but did not have sufficient Bloomberg news coverage to generate sentiment scores during their short membership period.

With only one quarter of index membership (approximately 63 trading days), these tickers would fall well below the 100 non-zero sentiment day minimum applied later in the pipeline. Their exclusion is expected and does not represent a data quality issue.

# Save Processed Sentiment File

In [11]:
df_sent.to_excel(r"..\..\Data\Main\Sentiment_Final.xlsx", index=False)
print(f"Saved Sentiment_Final.xlsx")
print(f"Shape: {df_sent.shape}")
print(f"Tickers: {len(sent_cols)}")
print(f"Zero sentiment tickers: {len(zero_tickers)}")
print(f"Tickers with data: {len(sent_cols) - len(zero_tickers)}")

Saved Sentiment_Final.xlsx
Shape: (1564, 1122)
Tickers: 561
Zero sentiment tickers: 14
Tickers with data: 547
